# Δημιουργία Κύβου Σύνοψης Διασφάλισης Εσόδων Τηλεπικοινωνιών με την PROC SUMMARY

## Περίληψη

Μια ομάδα διασφάλισης εσόδων σε έναν πάροχο κινητής τηλεφωνίας προσυγκεντρώνει έναν μήνα εγγραφών χρέωσης ανά συνδρομητή-ημέρα σε έναν συμπαγή κύβο σύνοψης, ώστε οι αναλυτές να μπορούν να διερευνήσουν σε βάθος τα διακανονισμένα έσοδα ανά περιοχή, επίπεδο πακέτου και τύπο κλήσης, χωρίς να χρειάζεται να σαρώσουν ξανά τον ακατέργαστο πίνακα γεγονότων. Η `PROC SUMMARY` συγκεντρώνει 100 εγγραφές συνδρομητή-ημέρας σε ένα πολυδιάστατο σύνολο κελιών: η λεπτομερέστερη διάσταση (περιοχή x επίπεδο πακέτου x τύπος κλήσης) γεμίζει 25 από τα 27 πιθανά κελιά, ενώ ονομασμένοι υποκύβοι δίνουν τα περιθώρια αθροίσματα που ζητούν συχνότερα οι αναλυτές. Σε αυτόν τον δείγμα μήνα, ο πάροχος διακανόνισε \$222.78 στις τρεις ενεργές περιοχές, με τον Νότο (\$97.44) και την Ανατολή (\$86.94) να αντιπροσωπεύουν μαζί το 83% των εσόδων, ενώ ο Βορράς υπολείπεται με \$38.40. Ο πλουσιότερος μεμονωμένος υποκύβος είναι η υπηρεσία Φωνής επιπέδου Προνομιακό (\$59.06 σε 18 εγγραφές), και η κατάταξη των κελιών ανά απόδοση-εγγραφή αναδεικνύει τα κελιά χρήσης δεδομένων ως τους στόχους υψηλότερης αξίας για έναν έλεγχο διαρροής εσόδων (κορυφαία απόδοση \$7.87/εγγραφή). Κάθε αριθμός παρακάτω διαβάζεται απευθείας από την εκτελεσμένη έξοδο.

## Πηγές Δεδομένων

| Σύνολο Δεδομένων | Γραμμές | Κλίμακα | Βασικές Μεταβλητές |
|---------|------|-------|---------------|
| `work.cdr_billing` | 100 | Μία γραμμή ανά σύνοψη χρήσης συνδρομητή-ημέρας | `region` (Ανατολή/Νότος/Βορράς), `plan_tier` (Προπληρωμένο/Βασικό/Προνομιακό), `call_type` (Φωνή/SMS/Δεδομένα), `device_class`, `bill_month`, `revenue`, `call_minutes`, `data_mb`, `subscriber_wt` |
| `work.cube_nway` | 25 | Μία γραμμή ανά μη κενό κελί (region x plan_tier x call_type) | `_FREQ_`, `rev_total`, `rev_mean`, `rev_max`, `min_total`, `data_total` |
| `work.cube_hier` | 22 | Μία γραμμή ανά κελί των ονομασμένων υποκύβων διερεύνησης | `_TYPE_`, `_FREQ_`, `rev_total` |

Όλα τα δεδομένα παράγονται εσωτερικά με `call streaminit()` / `rand()`· δεν απαιτούνται εξωτερικά αρχεία ή πρόσβαση σε δίκτυο. Αυτό το περιβάλλον λειτουργεί χωρίς άδεια, οπότε τα εγγεγραμμένα σύνολα δεδομένων περιορίζονται σε 100 παρατηρήσεις — το σενάριο έχει μέγεθος έτσι ώστε ο κύβος να είναι πλήρως γεμάτος εντός αυτού του ορίου.

## Από ακατέργαστες εγγραφές λεπτομερειών κλήσεων σε έναν διερευνήσιμο κύβο

Οι πάροχοι κινητής τηλεφωνίας διακανονίζουν έσοδα σε εκατομμύρια καθημερινά γεγονότα χρήσης. Για να επιτρέψουμε στους αναλυτές διασφάλισης εσόδων να απαντούν σε ερωτήματα όπως *"Ποια ήταν τα έσοδα φωνητικών κλήσεων επιπέδου Προνομιακό στην περιοχή Νότος τον προηγούμενο μήνα;"* χωρίς να σαρώνουν ξανά τον ακατέργαστο πίνακα γεγονότων κάθε φορά, **προσυγκεντρώνουμε** τα δεδομένα σε έναν συμπαγή κύβο σύνοψης.

Η `PROC SUMMARY` είναι το βασικό εργαλείο της Base SAS ακριβώς για αυτό: ομαδοποιεί έναν επίπεδο πίνακα γεγονότων κατά μία ή περισσότερες διαστάσεις `CLASS` και γράφει τα ζητούμενα στατιστικά σε ένα σύνολο δεδομένων εξόδου, επισημαίνοντας κάθε γραμμή με `_TYPE_` (ποιες διαστάσεις είναι ενεργές) και `_FREQ_` (εγγραφές πίσω από το κελί). Αυτό το σύνολο δεδομένων εξόδου *είναι* ένας πολυδιάστατος κύβος — η ίδια δομή συγκεντρωτικής άθροισης που θα εξέθετε ένα εργαλείο OLAP, υλοποιημένη ως ένα συνηθισμένο σύνολο δεδομένων SAS που μπορείτε να εκτυπώσετε, να συνενώσετε και να τεμαχίσετε.

Αυτό το notebook:

1. Δημιουργεί έναν ρεαλιστικό μήνα εγγραφών χρέωσης συνδρομητή-ημέρας.
2. Δημιουργεί τον κύβο λεπτομερέστερης διάστασης (περιοχή x επίπεδο πακέτου x τύπος κλήσης) με την `PROC SUMMARY NWAY`.
3. Υλοποιεί ονομασμένους **υποκύβους διερεύνησης** με τη δήλωση `TYPES`.
4. Προβάλλει τα έσοδα στη βάση συνδρομητών με ένα `WEIGHT`, διερευνά σε μία περιοχή, και κατατάσσει τα κελιά ανά απόδοση-εγγραφή για τριάζ διαρροής εσόδων.

## Βήμα 1 - Δημιουργία συνθετικών δεδομένων λεπτομερειών κλήσεων / χρέωσης

Κάθε γραμμή συνοψίζει τη χρήση μιας υπηρεσίας από έναν συνδρομητή σε μία ημέρα. Χρησιμοποιούμε την `call streaminit` για αναπαραγωγιμότητα και την `rand()` για να αντλήσουμε εύλογες κατανομές: τα έσοδα και η χρήση κλιμακώνονται με το επίπεδο πακέτου, τα έσοδα φωνής ακολουθούν τα χρεώσιμα λεπτά, και τα έσοδα δεδομένων ακολουθούν τα megabyte. Κάθε `RAND('table', ...)` λαμβάνει μία πιθανότητα ανά κατηγορία, ώστε κάθε περιοχή, επίπεδο και τύπος κλήσης να εμφανίζεται στο δείγμα των 100 εγγραφών. Προσαρτάται ένα μικρό βάρος έρευνας `subscriber_wt`, ώστε να μπορούμε αργότερα να επιδείξουμε ένα σταθμισμένο μέτρο.

In [1]:
ΔΕΔΟΜΕΝΑ work.cdr_billing;
    CALL streaminit(20260131);
    LENGTH region $20 plan_tier $30 call_type $20 device_class $16 bill_month $7;
    ΕΤΙΚΕΤΑ revenue       = "Διακανονισμένα Έσοδα (USD)"
          call_minutes  = "Χρεώσιμα Λεπτά Φωνής"
          data_mb       = "Όγκος Δεδομένων (MB)"
          subscriber_wt = "Βάρος Έρευνας Συνδρομητή"
          region        = "Περιοχή"
          plan_tier     = "Επίπεδο Πακέτου"
          call_type     = "Τύπος Κλήσης"
          device_class  = "Κατηγορία Συσκευής"
          bill_month    = "Μήνας Χρέωσης";

    ΕΠΑΝΑΛΗΨΗ i = 1 ΕΩΣ 100;
        /* --- Διαστάσεις: μία πιθανότητα ανά επίπεδο, με άθροισμα 1.0 --- */
        r = rand("table", 0.40, 0.33, 0.27);
        ΕΠΙΛΟΓΗ (r);
            ΟΤΑΝ (1) region = "Ανατολή";
            ΟΤΑΝ (2) region = "Νότος";
            ΑΛΛΟ region = "Βορράς";
        ΤΕΛΟΣ;

        p = rand("table", 0.30, 0.40, 0.30);
        ΕΠΙΛΟΓΗ (p);
            ΟΤΑΝ (1) plan_tier = "Προπληρωμένο";
            ΟΤΑΝ (2) plan_tier = "Βασικό";
            ΑΛΛΟ plan_tier = "Προνομιακό";
        ΤΕΛΟΣ;

        c = rand("table", 0.50, 0.30, 0.20);
        ΕΠΙΛΟΓΗ (c);
            ΟΤΑΝ (1) call_type = "Φωνή";
            ΟΤΑΝ (2) call_type = "SMS";
            ΑΛΛΟ call_type = "Δεδομένα";
        ΤΕΛΟΣ;

        ΕΑΝ rand("uniform") < 0.55 ΤΟΤΕ device_class = "Έξυπνο";
        ΑΛΛΙΩΣ device_class = "Απλό";

        bill_month = "2026-01";

        /* --- Μεγέθη, κλιμακωμένα κατά επίπεδο και υπηρεσία --- */
        tier_mult = (plan_tier = "Προπληρωμένο")*0.7
                  + (plan_tier = "Βασικό")*1.0
                  + (plan_tier = "Προνομιακό")*1.7;

        call_minutes = round((call_type = "Φωνή")
                       * rand("gamma", 2.0) * 18 * tier_mult, 0.1);
        data_mb      = round((call_type = "Δεδομένα")
                       * rand("gamma", 1.5) * 220 * tier_mult, 0.1);

        base_rev = 0.05*call_minutes + 0.010*data_mb
                 + (call_type = "SMS") * rand("poisson", 30) * 0.03;
        revenue = round(base_rev * (0.85 + 0.30*rand("uniform")), 0.01);

        subscriber_wt = round(0.8 + rand("uniform")*1.6, 0.01);

        ΕΞΟΔΟΣ;
    ΤΕΛΟΣ;
    ΑΦΑΙΡΕΣΗ i r p c tier_mult base_rev;
ΕΚΤΕΛΕΣΗ;

ΔΙΑΔΙΚΑΣΙΑ PRINT ΔΕΔΟΜΕΝΑ=work.cdr_billing(obs=8) ΕΤΙΚΕΤΑ noobs;
    TITLE "Δείγμα Εγγραφών Χρέωσης Συνδρομητή-Ημέρας";
ΕΚΤΕΛΕΣΗ;

                                       Δείγμα Εγγραφών Χρέωσης Συνδρομητή-Ημέρας                                        

       Περιοχή                Επίπεδο Πακέτου             Τύπος Κλήσης                   Κατηγορία Συσκευής              Μήνας Χρέωσης                    Χρεώσιμα Λεπτά Φωνής                Όγκος Δεδομένων (MB)                     Διακανονισμένα Έσοδα (USD)                        Βάρος Έρευνας Συνδρομητή
Βορράς          Προνομιακό                     SMS                      Έξυπνο                               2026-01                                                         0                                   0                                           0.67                                            1.13
Νότος           Προπληρωμένο                   SMS                      Απλό                                 2026-01                                                         0                                   0                                           0.94         


NOTE: DATA work.cdr_billing


NOTE: Wrote work.cdr_billing (100 rows, 9 columns).
NOTE: DATA elapsed:
  wall  0.04 seconds
  cpu   0.04 seconds
NOTE: PROC PRINT data=work.cdr_billing

NOTE: PROC PRINT completed: 8 observations printed, 9 variables


## Βήμα 2 - Δημιουργία του κύβου λεπτομερέστερης διάστασης με την PROC SUMMARY NWAY

Η `NWAY` διατηρεί μόνο τον μοναδικό πιο λεπτομερή συνδυασμό όλων των διαστάσεων `CLASS` - εδώ κάθε γεμάτο κελί (region x plan_tier x call_type). Για κάθε κελί αποθηκεύουμε τα έσοδα `SUM`, `MEAN` και `MAX`, καθώς και τα συνολικά λεπτά και megabyte, ώστε ένας αναλυτής να μπορεί να διαβάσει τα συνολικά έσοδα ανά κελί, να προκύψει ο μέσος όρος ανά εγγραφή, και να εντοπίσει τη μεγαλύτερη μεμονωμένη χρέωση. Το `_FREQ_` καταγράφει πόσες συνδρομητή-ημέρες βρίσκονται πίσω από κάθε κελί. Αφαιρούμε το `_TYPE_` εδώ επειδή, στη διάσταση NWAY, κάθε γραμμή έχει τον ίδιο τύπο.

In [2]:
ΔΙΑΔΙΚΑΣΙΑ summary ΔΕΔΟΜΕΝΑ=work.cdr_billing NWAY;
    ΚΛΑΣΗ region plan_tier call_type;
    ΜΕΤΑΒΛΗΤΗ revenue call_minutes data_mb;
    ΕΞΟΔΟΣ out=work.cube_nway(ΑΦΑΙΡΕΣΗ=_type_)
           sum(revenue)=rev_total  mean(revenue)=rev_mean  MAX(revenue)=rev_max
           sum(call_minutes)=min_total
           sum(data_mb)=data_total;
ΕΚΤΕΛΕΣΗ;

ΔΙΑΔΙΚΑΣΙΑ PRINT ΔΕΔΟΜΕΝΑ=work.cube_nway(obs=12) ΕΤΙΚΕΤΑ noobs;
    TITLE "Κελιά κύβου NWAY (περιοχή * επίπεδο πακέτου * τύπος κλήσης)";
    ΕΤΙΚΕΤΑ region="Περιοχή" plan_tier="Επίπεδο Πακέτου" call_type="Τύπος Κλήσης"
          _freq_="Πλήθος Εγγραφών"
          rev_total="Σύνολο Εσόδων" rev_mean="Μέσος Όρος Εσόδων" rev_max="Μέγιστο Έσοδο"
          min_total="Συνολικά Λεπτά Κλήσεων" data_total="Σύνολο MB Δεδομένων";
    ΜΟΡΦΗ rev_total rev_mean rev_max min_total data_total comma10.2;
ΕΚΤΕΛΕΣΗ;

                              Κελιά κύβου NWAY (περιοχή * επίπεδο πακέτου * τύπος κλήσης)                               

       Περιοχή                Επίπεδο Πακέτου             Τύπος Κλήσης                Πλήθος Εγγραφών              Σύνολο Εσόδων                 Μέσος Όρος Εσόδων              Μέγιστο Έσοδο                      Συνολικά Λεπτά Κλήσεων                 Σύνολο MB Δεδομένων
Ανατολή         Βασικό                         SMS                                                  4                       4.07                              1.02                       1.24                                        0.00                                0.00
Ανατολή         Βασικό                         Δεδομένα                                             4                      18.17                              4.54                       8.05                                        0.00                            1,807.90
Ανατολή         Βασικό                         Φωνή                 


NOTE: PROC MEANS
NOTE: Output dataset work.cube_nway has 25 observations and 9 variables.
NOTE: PROC MEANS statement used.
NOTE: PROC PRINT data=work.cube_nway

NOTE: PROC PRINT completed: 12 observations printed, 9 variables


## Βήμα 3 - Υλοποίηση ονομασμένων υποκύβων διερεύνησης με την TYPES

Ένας κύβος OLAP προαποθηκεύει τις συγκεντρωτικές αθροίσεις που διερευνούν συχνότερα οι αναλυτές. Η δήλωση `TYPES` κάνει ακριβώς αυτό: κάθε όρος ζητά από την `PROC SUMMARY` να παράγει έναν υποκύβο. Ζητάμε το γενικό σύνολο `()`, το περιθώριο άθροισμα της `region`, και τους διπλής κατεύθυνσης υποκύβους `region*plan_tier` και `call_type*plan_tier` - τα μονοπάτια διερεύνησης που θα εξέθετε ένα ταμπλό εσόδων. Η SAS επισημαίνει κάθε υποκύβο με έναν κωδικό `_TYPE_` (μια μάσκα bit πάνω από τη λίστα `CLASS`), ώστε ένα μοναδικό σύνολο δεδομένων εξόδου να φέρει κάθε επίπεδο της ιεραρχίας.

In [3]:
ΔΙΑΔΙΚΑΣΙΑ summary ΔΕΔΟΜΕΝΑ=work.cdr_billing;
    ΚΛΑΣΗ region plan_tier call_type;
    ΜΕΤΑΒΛΗΤΗ revenue;
    TYPES () region region*plan_tier call_type*plan_tier;
    ΕΞΟΔΟΣ out=work.cube_hier
           sum(revenue)=rev_total;
ΕΚΤΕΛΕΣΗ;

ΔΙΑΔΙΚΑΣΙΑ PRINT ΔΕΔΟΜΕΝΑ=work.cube_hier ΕΤΙΚΕΤΑ noobs;
    TITLE "Υποκύβοι συγκεντρωτικής άθροισης: γενικό σύνολο, περιοχή, περιοχή*επίπεδο, τύπος_κλήσης*επίπεδο";
    ΜΕΤΑΒΛΗΤΗ _type_ region plan_tier call_type _freq_ rev_total;
    ΕΤΙΚΕΤΑ _type_="Τύπος Υποκύβου" region="Περιοχή" plan_tier="Επίπεδο Πακέτου"
          call_type="Τύπος Κλήσης" _freq_="Πλήθος Εγγραφών" rev_total="Σύνολο Εσόδων";
    ΜΟΡΦΗ rev_total comma10.2;
ΕΚΤΕΛΕΣΗ;

            Υποκύβοι συγκεντρωτικής άθροισης: γενικό σύνολο, περιοχή, περιοχή*επίπεδο, τύπος_κλήσης*επίπεδο             

             Τύπος Υποκύβου         Περιοχή                Επίπεδο Πακέτου             Τύπος Κλήσης                Πλήθος Εγγραφών              Σύνολο Εσόδων
                          0                                                                                                    100                     222.78
                          3                  Βασικό                         SMS                                                  8                       8.03
                          3                  Βασικό                         Δεδομένα                                             8                      38.06
                          3                  Βασικό                         Φωνή                                                20                      42.33
                          3                  Προνομιακό                     SMS         


NOTE: PROC MEANS
NOTE: Output dataset work.cube_hier has 22 observations and 6 variables.
NOTE: PROC MEANS statement used.
NOTE: PROC PRINT data=work.cube_hier

NOTE: PROC PRINT completed: 22 observations printed, 6 variables


## Βήμα 4 - Σταθμισμένη προβολή, διερεύνηση περιοχής, και τριάζ διαρροής

Τρεις αναγνώσεις που μια ομάδα διασφάλισης εσόδων πραγματικά εκτελεί πάνω στον κύβο:

- **Σταθμισμένη προβολή.** Η προσάρτηση `WEIGHT=subscriber_wt` σε μια σύνοψη `region*plan_tier` αναφέρει έσοδα κλιμακωμένα στην πλήρη βάση συνδρομητών που αντιπροσωπεύει το δείγμα, αντί για τις ακατέργαστες δειγματισμένες γραμμές.
- **Διερεύνηση.** Το φιλτράρισμα του κύβου NWAY σε μία περιοχή επεκτείνει έναν μοναδικό κλάδο της ιεραρχίας - εδώ τον Νότο - στη λεπτομέρειά του ανά επίπεδο και υπηρεσία.
- **Τριάζ διαρροής.** Η ταξινόμηση των κελιών κατά μέσα έσοδα ανά εγγραφή αναδεικνύει τα κελιά υψηλότερης απόδοσης· τα κελιά με χαμηλή συχνότητα και υψηλή απόδοση είναι ακριβώς αυτά που εξετάζει ένας έλεγχος για εσφαλμένα τιμολογημένα ή διαρρέοντα έσοδα.

In [4]:
/* Σταθμισμένα έσοδα προβεβλημένα στη βάση συνδρομητών */
ΔΙΑΔΙΚΑΣΙΑ summary ΔΕΔΟΜΕΝΑ=work.cdr_billing NWAY;
    ΚΛΑΣΗ region plan_tier;
    ΜΕΤΑΒΛΗΤΗ revenue;
    ΒΑΡΟΣ subscriber_wt;
    ΕΞΟΔΟΣ out=work.cube_wt(ΑΦΑΙΡΕΣΗ=_type_ _freq_)
           sum(revenue)=rev_weighted  n(revenue)=records;
ΕΚΤΕΛΕΣΗ;

ΔΙΑΔΙΚΑΣΙΑ PRINT ΔΕΔΟΜΕΝΑ=work.cube_wt ΕΤΙΚΕΤΑ noobs;
    TITLE "Σταθμισμένα έσοδα ανά περιοχή * επίπεδο πακέτου (προβεβλημένα στη βάση συνδρομητών)";
    ΕΤΙΚΕΤΑ region="Περιοχή" plan_tier="Επίπεδο Πακέτου"
          rev_weighted="Σταθμισμένα Έσοδα" records="Εγγραφές";
    ΜΟΡΦΗ rev_weighted comma10.2;
ΕΚΤΕΛΕΣΗ;

/* Διερεύνηση στον κλάδο της περιοχής Νότος */
ΔΙΑΔΙΚΑΣΙΑ PRINT ΔΕΔΟΜΕΝΑ=work.cube_nway ΕΤΙΚΕΤΑ noobs;
    ΟΠΟΥ region = "Νότος";
    TITLE "Διερεύνηση στον Νότο: κελιά εσόδων ανά επίπεδο και τύπο κλήσης";
    ΜΕΤΑΒΛΗΤΗ plan_tier call_type _freq_ rev_total rev_mean;
    ΕΤΙΚΕΤΑ plan_tier="Επίπεδο Πακέτου" call_type="Τύπος Κλήσης" _freq_="Πλήθος Εγγραφών"
          rev_total="Σύνολο Εσόδων" rev_mean="Μέσος Όρος Εσόδων";
    ΜΟΡΦΗ rev_total rev_mean comma10.2;
ΕΚΤΕΛΕΣΗ;

/* Κατάταξη των κελιών ανά απόδοση-εγγραφή για τριάζ διαρροής */
ΔΙΑΔΙΚΑΣΙΑ SORT ΔΕΔΟΜΕΝΑ=work.cube_nway out=work.cube_ranked;
    ΚΑΤΑ DESCENDING rev_mean;
ΕΚΤΕΛΕΣΗ;

ΔΙΑΔΙΚΑΣΙΑ PRINT ΔΕΔΟΜΕΝΑ=work.cube_ranked(obs=6) ΕΤΙΚΕΤΑ noobs;
    TITLE "Κελιά με το υψηλότερο μέσο έσοδο (απόδοση ανά εγγραφή)";
    ΜΕΤΑΒΛΗΤΗ region plan_tier call_type _freq_ rev_mean rev_max;
    ΕΤΙΚΕΤΑ region="Περιοχή" plan_tier="Επίπεδο Πακέτου" call_type="Τύπος Κλήσης"
          _freq_="Πλήθος Εγγραφών" rev_mean="Μέσος Όρος Εσόδων" rev_max="Μέγιστο Έσοδο";
    ΜΟΡΦΗ rev_mean rev_max comma10.2;
ΕΚΤΕΛΕΣΗ;

                  Σταθμισμένα έσοδα ανά περιοχή * επίπεδο πακέτου (προβεβλημένα στη βάση συνδρομητών)                   

       Περιοχή                Επίπεδο Πακέτου                  Σταθμισμένα Έσοδα          Εγγραφές
Ανατολή         Βασικό                                                     50.85                15
Ανατολή         Προνομιακό                                                 59.59                12
Ανατολή         Προπληρωμένο                                               29.77                11
Βορράς          Βασικό                                                     18.30                 7
Βορράς          Προνομιακό                                                 22.42                 7
Βορράς          Προπληρωμένο                                               20.56                 9
Νότος           Βασικό                                                     58.63                14
Νότος           Προνομιακό                                                 56.29      


NOTE: PROC MEANS
NOTE: Output dataset work.cube_wt has 9 observations and 4 variables.
NOTE: PROC MEANS statement used.
NOTE: PROC PRINT data=work.cube_wt

NOTE: PROC PRINT completed: 9 observations printed, 4 variables
NOTE: PROC PRINT data=work.cube_nway

NOTE: PROC PRINT completed: 9 observations printed, 5 variables
NOTE: PROC SORT data=work.cube_nway

NOTE: Unlicensed mode - input limited to 100 observations.
NOTE: Read 25 rows from work.cube_nway.
NOTE: Wrote work.cube_ranked (25 rows, 9 columns).
NOTE: PROC SORT statement used.
NOTE: PROC PRINT data=work.cube_ranked

NOTE: PROC PRINT completed: 6 observations printed, 6 variables


## Ερμηνεία των αποτελεσμάτων

Ο κύβος σύνοψης μετατρέπει 100 ακατέργαστες γραμμές συνδρομητή-ημέρας σε ένα συμπαγές σύνολο προσυγκεντρωμένων κελιών που απαντούν σε ερωτήματα διερεύνησης άμεσα, χωρίς να σαρώνουν ξανά τον πίνακα γεγονότων:

- **Πού βρίσκονται τα έσοδα.** Το περιθώριο άθροισμα της `region` (`_TYPE_=4`) δείχνει ότι ο Νότος διακανόνισε \$97.44 και η Ανατολή \$86.94 - μαζί το 83% του μήνα των \$222.78 - ενώ ο Βορράς συνεισέφερε \$38.40. Μέσα στον υποκύβο `call_type*plan_tier` (`_TYPE_=3`), η Φωνή επιπέδου Προνομιακό είναι το πλουσιότερο μεμονωμένο κελί με \$59.06 σε 18 εγγραφές, με τη Φωνή επιπέδου Βασικό αμέσως μετά στα \$42.33.
- **Σταθμισμένη προβολή.** Η εφαρμογή του βάρους έρευνας ανακατατάσσει την κατάταξη προς πακέτα των οποίων οι συνδρομητές φέρουν μεγαλύτερο βάρος: Ανατολή-Προνομιακό (\$59.59) και Νότος-Βασικό (\$58.63) οδηγούν τα προβεβλημένα έσοδα `region*plan_tier`, μια διαφορετική εικόνα από τα μη σταθμισμένα σύνολα περιοχής, και μια υπενθύμιση να αναφέρουμε προβεβλημένα και όχι δειγματισμένα έσοδα όταν εκτιμούμε την έκθεση.
- **Απόδοση ανά εγγραφή και τριάζ διαρροής.** Η κατάταξη των κελιών NWAY κατά `rev_mean` φέρνει στην κορυφή τα κελιά χρήσης δεδομένων - Βορράς-Βασικό-Δεδομένα (\$7.87/εγγραφή) και Νότος-Προνομιακό-Δεδομένα (\$5.96/εγγραφή) - επιβεβαιώνοντας ότι η χρήση με έντονη κατανάλωση δεδομένων οδηγεί τα υψηλότερα έσοδα ανά εγγραφή. Οι ηγέτες χαμηλής συχνότητας (μία ή δύο εγγραφές) είναι ακριβώς τα κελιά για τα οποία ένας αναλυτής διασφάλισης εσόδων θα ανέσυρε τις υποκείμενες εγγραφές λεπτομερειών κλήσεων, για να επιβεβαιώσει ότι η υψηλή χρέωση είναι σωστά τιμολογημένη και όχι σφάλμα.

Για μια ομάδα διασφάλισης εσόδων, αυτός ο κύβος αποτελεί το θεμέλιο για ταμπλό διακύμανσης: σύγκριση των διακανονισμένων εσόδων με τα αναμενόμενα έσοδα τιμοκαταλόγου ανά κελί (περιοχή, επίπεδο, τύπος κλήσης), και τα κελιά των οποίων ο μέσος όρος ή το σταθμισμένο σύνολο αποκλίνουν περισσότερο γίνονται οι υποθέσεις διαρροής που αξίζει να ελεγχθούν. Επειδή ολόκληρη η δομή είναι ένα συνηθισμένο σύνολο δεδομένων SAS, ο κύβος του επόμενου μήνα μπορεί να ενωθεί, να αφαιρεθεί, ή να συνενωθεί με έναν τιμοκατάλογο χρησιμοποιώντας τα ίδια εργαλεία Base SAS.